In [3]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
restaurant_path = '/content/drive/MyDrive/DS5500 Capstone/yelp_academic_dataset_business.json'
review_path = '/content/drive/MyDrive/DS5500 Capstone/yelp_academic_dataset_review.json'
user_path = '/content/drive/MyDrive/DS5500 Capstone/yelp_academic_dataset_user.json'

In [5]:
restaurant_df = pd.read_json(restaurant_path, lines = True)
review_df = pd.read_json(review_path, lines = True)
user_df = pd.read_json(user_path, lines = True)

In [6]:
restaurant_df.head()

              business_id                      name  \
0  Pns2l4eNsfO8kk83dixA6A  Abby Rappoport, LAC, CMQ   
1  mpf3x-BjTdTEA3yCZrAYPw             The UPS Store   
2  tUFrWirKiKi_TAnsVWINQQ                    Target   
3  MTSW4McQd7CbVtyjqoe9mw        St Honore Pastries   
4  mWMc6_wTdE0EUBKIGXDVfA  Perkiomen Valley Brewery   

                           address           city state postal_code  \
0           1616 Chapala St, Ste 2  Santa Barbara    CA       93101   
1  87 Grasso Plaza Shopping Center         Affton    MO       63123   
2             5255 E Broadway Blvd         Tucson    AZ       85711   
3                      935 Race St   Philadelphia    PA       19107   
4                    101 Walnut St     Green Lane    PA       18054   

    latitude   longitude  stars  review_count  is_open  \
0  34.426679 -119.711197    5.0             7        0   
1  38.551126  -90.335695    3.0            15        1   
2  32.223236 -110.880452    3.5            22        0   
3  39.9555

In [7]:
my_state = 'CA'
restaurant_ids = set(restaurant_df[restaurant_df['state'] == my_state]['business_id'])

In [8]:
CA_Restaurants = restaurant_df[restaurant_df['state'] == my_state]

In [9]:
CA_Restaurants = CA_Restaurants.reset_index(drop=True)
CA_Restaurants.to_csv("/content/drive/MyDrive/DS5500 Capstone/CA_Restaurants.csv")

In [10]:
FOOD_CATEGORIES = {
    # Core
    "Restaurants", "Food",
    # Cuisine types
    "American (New)", "American (Traditional)", "Arabic", "Argentine",
    "Asian Fusion", "Australian", "Barbeque", "Basque", "Belgian",
    "Brazilian", "British", "Buffets", "Burgers", "Cafeteria",
    "Cajun/Creole", "Cantonese", "Caribbean", "Cheesesteaks",
    "Chicken Shop", "Chicken Wings", "Chinese", "Comfort Food",
    "Cuban", "Dim Sum", "Diners", "Ethiopian", "Falafel",
    "Fast Food", "Fish & Chips", "Fondue", "French", "German",
    "Greek", "Hawaiian", "Himalayan/Nepalese", "Hot Dogs", "Hot Pot",
    "Indian", "Indonesian", "Irish", "Italian", "Japanese",
    "Kebab", "Korean", "Latin American", "Lebanese",
    "Mediterranean", "Mexican", "Middle Eastern", "Modern European",
    "Mongolian", "Moroccan", "New Mexican Cuisine", "Noodles",
    "Pakistani", "Pan Asian", "Persian/Iranian", "Peruvian",
    "Pizza", "Poke", "Ramen", "Salad", "Salvadoran", "Sandwiches",
    "Scandinavian", "Seafood", "Smokehouse", "Soul Food", "Soup",
    "Southern", "Spanish", "Steakhouses", "Sushi Bars",
    "Szechuan", "Tacos", "Taiwanese", "Tapas/Small Plates",
    "Tex-Mex", "Thai", "Turkish", "Tuscan", "Vietnamese", "Wraps",
    # Drink & casual
    "Bars", "Beer Bar", "Beer Gardens", "Breweries", "Brewpubs",
    "Cideries", "Cocktail Bars", "Coffee & Tea", "Coffeeshops",
    "Distilleries", "Dive Bars", "Food Court", "Food Stands",
    "Food Trucks", "Gastropubs", "Gay Bars", "Hookah Bars",
    "Irish Pub", "Jazz & Blues", "Juice Bars & Smoothies",
    "Karaoke", "Lounges", "Piano Bars", "Pop-Up Restaurants",
    "Pubs", "Speakeasies", "Sports Bars", "Tapas Bars",
    "Tea Rooms", "Tiki Bars", "Whiskey Bars", "Wine Bars",
    # Food subtypes
    "Acai Bowls", "Bagels", "Bakeries", "Bubble Tea", "Butcher",
    "Candy Stores", "Cheese Shops", "Chocolatiers & Shops",
    "Creperies", "Cupcakes", "Custom Cakes", "Delicatessen",
    "Delis", "Desserts", "Do-It-Yourself Food", "Donuts",
    "Ethnic Food", "Ethnic Grocery", "Food Delivery Services",
    "Fruits & Veggies", "Gelato", "Gluten-Free", "Halal",
    "Health Markets", "Hong Kong Style Cafe", "Ice Cream & Frozen Yogurt",
    "International Grocery", "Live/Raw Food", "Macarons",
    "Meat Shops", "Organic Stores", "Pasta Shops",
    "Patisserie/Cake Shop", "Personal Chefs", "Pretzels",
    "Shaved Ice", "Specialty Food", "Street Vendors",
    "Vegan", "Vegetarian", "Cafes", "Caterers",
    "Breakfast & Brunch", "Grocery",
    # Ambiguous but food-related
    "Themed Cafes", "Internet Cafes",
}

def has_food_category(cat_str):
    if pd.isna(cat_str):
        return False
    cats = {c.strip() for c in cat_str.split(",")}
    return bool(cats & FOOD_CATEGORIES)

df_food = CA_Restaurants[CA_Restaurants["categories"].apply(has_food_category)].copy()
print(f"After food filter: {len(df_food)}")
"""
catering_only = df["categories"].str.contains("Caterers", na=False) & \
            ~df["categories"].str.contains("Restaurants", na=False)
df = df[~catering_only].reset_index(drop=True)
print(f"After catering filter: {len(df)}")

# ─────────────────────────────────────────
# 3. OPTIONAL: KEEP ONLY OPEN BUSINESSES
# ─────────────────────────────────────────
df_food = df_food[df_food["is_open"] == 1].copy()
print(f"After open filter: {len(df_food)}")

# ─────────────────────────────────────────
# 4. CLEAN COLUMNS
# ─────────────────────────────────────────
# Fix postal_code float → string
df_food["postal_code"] = df_food["postal_code"].apply(
    lambda x: str(int(x)) if pd.notna(x) else ""
)

# Reset index so row index = ChromaDB document ID
df_food = df_food.reset_index(drop=True)
print(f"\nFinal restaurant count: {len(df_food)}")
print(f"Cities covered: {df_food['city'].nunique()}")
print(f"\nTop 10 cities:\n{df_food['city'].value_counts().head(10)}")
print(f"\nStar distribution:\n{df_food['stars'].value_counts().sort_index()}")

# ─────────────────────────────────────────
# 5. PARSE ATTRIBUTES & EXTRACT PRICE
# ─────────────────────────────────────────
def parse_dict_field(val):
    if isinstance(val, dict):
        return val
    try:
        return ast.literal_eval(str(val))
    except:
        return {}

def extract_price(attrs_str):
    attrs = parse_dict_field(attrs_str)
    val = str(attrs.get("RestaurantsPriceRange2", "")).strip("'\" ")
    try:
        return int(val)
    except:
        return None

df_food["price"] = df_food["attributes"].apply(extract_price)
print(f"\nPrice distribution:\n{df_food['price'].value_counts(dropna=False).sort_index()}")
"""

After food filter: 1662


'\ncatering_only = df["categories"].str.contains("Caterers", na=False) &             ~df["categories"].str.contains("Restaurants", na=False)\ndf = df[~catering_only].reset_index(drop=True)\nprint(f"After catering filter: {len(df)}")\n\n# ─────────────────────────────────────────\n# 3. OPTIONAL: KEEP ONLY OPEN BUSINESSES\n# ─────────────────────────────────────────\ndf_food = df_food[df_food["is_open"] == 1].copy()\nprint(f"After open filter: {len(df_food)}")\n\n# ─────────────────────────────────────────\n# 4. CLEAN COLUMNS\n# ─────────────────────────────────────────\n# Fix postal_code float → string\ndf_food["postal_code"] = df_food["postal_code"].apply(\n    lambda x: str(int(x)) if pd.notna(x) else ""\n)\n\n# Reset index so row index = ChromaDB document ID\ndf_food = df_food.reset_index(drop=True)\nprint(f"\nFinal restaurant count: {len(df_food)}")\nprint(f"Cities covered: {df_food[\'city\'].nunique()}")\nprint(f"\nTop 10 cities:\n{df_food[\'city\'].value_counts().head(10)}")\nprin

In [25]:
df_food.head()
restaurant_ids = set(df_food['business_id'])

In [26]:
review_df.head()

                review_id                 user_id             business_id  \
0  KU_O5udG6zpxOg-VcAEodg  mh_-eMZ6K5RLWhZyISBhwA  XQfwVwDr-v0ZS3_CbbE5Xw   
1  BiTunyQ73aT9WBnpR9DZGw  OyoGAe7OKpv6SyGZT5g77Q  7ATYjTIgM3jUlt4UM3IypQ   
2  saUsX_uimxRlCVr67Z4Jig  8g_iMtfSiwikVnbP2etR0A  YjUWPpI6HXG530lwP-fb2A   
3  AqPFMleE6RsU23_auESxiA  _7bHUi9Uuf5__HHc_Q8guQ  kxX2SOes4o-D3ZQBkiMRfA   
4  Sx8TMOWLNuJBWer-0pcmoA  bcjbaE6dDog4jkNY91ncLQ  e4Vwtrqf-wpJfwesgvdgxQ   

   stars  useful  funny  cool  \
0      3       0      0     0   
1      5       1      0     1   
2      3       0      0     0   
3      5       1      0     1   
4      4       1      0     1   

                                                text                date  
0  If you decide to eat here, just be aware it is... 2018-07-07 22:09:11  
1  I've taken a lot of spin classes over the year... 2012-01-03 15:28:18  
2  Family diner. Had the buffet. Eclectic assortm... 2014-02-05 20:30:30  
3  Wow!  Yummy, different,  delicious.

In [27]:
user_ids = set(review_df[review_df['business_id'].isin(restaurant_ids)]['user_id'])

In [28]:
user_df[user_df['user_id'].isin(user_ids)].reset_index(drop=True).to_csv("/content/drive/MyDrive/DS5500 Capstone/CA_Users.csv")

In [29]:
#Rows are user ids
#Columns are restaurant ids

relevant_reviews_df = review_df[review_df['business_id'].isin(restaurant_ids)]


In [30]:
relevant_reviews_df.head()

                 review_id                 user_id             business_id  \
9   pUycOfUwM8vqX7KjRRhUEA  59MxRhNVhU9MYndMkz0wtw  gebiRewfieSdtt17PTW6Zg   
23  eCiWBf1CJ0Zdv1uVarEhhw  OhECKhQEexFypOMY6kypRw  vC2qm1y3Au5czBtbhc-DNw   
31  YbMyvlDA2W3Py5lTz8VK-A  4hBhtCSgoxkrFgHa4YAD-w  bbEXAEFr4RYHLlZ-HFssTA   
35  L0jv8c2FbpWSlfNC6bbUEA  bFPdtzu11Oi0f92EAcjqmg  IDtLPgUrqorrpqSLdfMhZQ   
61  4zopEEPqfwm-c_FNpeHZYw  JYYYKt6TdVA4ng9lLcXt_g  SZU9c8V2GuREDN5KgyHFJw   

    stars  useful  funny  cool  \
9       3       0      0     0   
23      4       0      0     0   
31      5       0      0     0   
35      5       0      0     0   
61      5       0      0     0   

                                                 text                date  
9   Had a party of 6 here for hibachi. Our waitres... 2016-07-25 07:31:06  
23  Yes, this is the only sushi place in town. How... 2013-09-04 03:48:20  
31  Great burgers,fries and salad!  Burgers have a... 2017-01-02 03:17:34  
35  What a great addit

In [31]:
relevant_reviews_df.reset_index(drop=True).to_csv("/content/drive/MyDrive/DS5500 Capstone/CA_Reviews.csv")

In [32]:
review_matrix = relevant_reviews_df.pivot_table(index='user_id', columns='business_id', values='stars', aggfunc='first', fill_value=0)

In [33]:
from scipy.sparse import csr_matrix

sparse_review_matrix = csr_matrix(review_matrix.values)

In [34]:
len(review_matrix.columns)

1662

In [35]:
!pip install implicit
from scipy.sparse import csr_matrix
from implicit import als
import numpy as np


# 2. Train ALS model
model = als.AlternatingLeastSquares(
    factors=50,        # latent dimensions
    regularization=0.1,
    iterations=20,
    use_gpu=True
)
model.fit(sparse_review_matrix)

# 3. Get recommendations for a user
user_idx = 0  # index into your matrix
ids, scores = model.recommend(
    user_idx,
    sparse_review_matrix[user_idx],
    N=10                    # top 10 recommendations
    #filter_already_liked=True
)

# Map indices back to restaurant ids
recommended = review_matrix.columns[ids]

In [36]:
list(recommended)

['r2IhvKZQ_wLR5mLBnPOilg',
 'iMTjejk6apJKzCukZnDw5A',
 'l-rGtJt0E7PAklT0IK7oFQ',
 'H-1qpp_77KggOAr9htUrEw',
 'S3QHy1sshUeZwXOYviVsXQ',
 'wFES5bGDiPANW13hNHnXOQ',
 'U2hkeMI-q4cS35QolJYN0A',
 'aFhPwy7OhyKah4cJY2QdEQ',
 'skY6r8WAkYqpV7_TxNm23w',
 'b5hxsRw76UUXcotxlCipuQ']

In [37]:
restaurant_df[restaurant_df['business_id'].isin(list(recommended))]

                   business_id                                            name  \
5721    H-1qpp_77KggOAr9htUrEw                           Cava Restaurant & Bar   
8679    wFES5bGDiPANW13hNHnXOQ                     Sakana Sushi Bar & Japanese   
12105   S3QHy1sshUeZwXOYviVsXQ                                       Tre Lune   
46363   U2hkeMI-q4cS35QolJYN0A                                  Arigato Sushi   
89510   aFhPwy7OhyKah4cJY2QdEQ                         Stonehouse Restaurant   
120988  b5hxsRw76UUXcotxlCipuQ      Los Arroyos Mexican Restaurant & Take Out   
124421  iMTjejk6apJKzCukZnDw5A                 Jeannine's Bakery & Restaurant   
138710  l-rGtJt0E7PAklT0IK7oFQ  Four Seasons Resort The Biltmore Santa Barbara   
139786  skY6r8WAkYqpV7_TxNm23w                     Boathouse at Hendry's Beach   
148705  r2IhvKZQ_wLR5mLBnPOilg                                  The Honor Bar   

                               address           city state  stars  
5721             1212 Coast Village

In [41]:
from scipy.sparse import csr_matrix
!pip install implicit
from scipy.sparse import csr_matrix
from implicit import als
import numpy as np

def train_collab_filter(review_matrix, factors, regularization, iterations):
  sparse_review_matrix = csr_matrix(review_matrix.values)
  #Train ALS model
  model = als.AlternatingLeastSquares(
      factors=factors,        # latent dimensions
      regularization=regularization,
      iterations=iterations,
      use_gpu=True
  )
  model.fit(sparse_review_matrix)

  return model

def predict_collab_filter(restaurant_df, model, review_matrix, N, user_idx):
  sparse_review_matrix = csr_matrix(review_matrix.values)
  ids, scores = model.recommend(
      user_idx,
      sparse_review_matrix[user_idx],
      N=N                    # top N recommendations
  )

  # Map indices back to restaurant ids
  recommended = review_matrix.columns[ids]
  return restaurant_df[restaurant_df['business_id'].isin(list(recommended))]

In [39]:
my_model = train_collab_filter(review_matrix, 50, .1, 20)

In [42]:
predict_collab_filter(restaurant_df, my_model, review_matrix, 5, 15)

                  business_id                      name  \
7942   uO39--k_hrCFgZh-Bl8m8A       Taqueria Cuernavaca   
32021  4uspKB1d3pGdgjOClA-UkQ        Cajun Kitchen Cafe   
47944  3fpAmsSuEFNF29UUPpgwlw         Spudnuts & Bagels   
54486  E_ac8Q20OJD9L8cafCBKaQ        Milk & Honey Tapas   
80309  DLTwr11KNMA1hEN7OrkY2A  Mac's Fish and Chip Shop   

                         address           city state  stars  
7942           201 W Carrillo St  Santa Barbara    CA    4.5  
32021  1924 De La Vina St, Ste A  Santa Barbara    CA    4.0  
47944              3629 State St  Santa Barbara    CA    4.5  
54486            30 W Anapamu St  Santa Barbara    CA    4.0  
80309               503 State St  Santa Barbara    CA    3.5  